# Create Tools for Agents

In [3]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool 

# Load Environment Variables 
load_dotenv()

True

In [4]:
model = init_chat_model(model="groq:openai/gpt-oss-120b")

In [5]:
# Docstring after Function Definition Provides Tool Information to the Model
@tool
def get_weather(location: str) -> str:
    """Get Weather at a Location"""  
    return f"It's sunny at {location}"

model_with_tools = model.bind_tools([get_weather])  # Way 1 - To Create a Tool for the Model

In [7]:
response = model_with_tools.invoke("What's the weather like in Boston?")

print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks current weather in Boston. Need to call get_weather function.', 'tool_calls': [{'id': 'fc_e33a4236-80da-4a01-8309-6d55b7014db2', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 126, 'total_tokens': 168, 'completion_time': 0.087867189, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.004927269, 'prompt_tokens_details': None, 'queue_time': 0.15921311, 'total_time': 0.092794458}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f2f3c-ec93-70d3-9ada-be40c18d2bc7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_e33a4236-80da-4a01-8309-6d55b7014db2', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={

In [ ]:
# Create an Agent and add tool to the tools attribute
agent = create_agent(
    model=model,
    tools=[get_weather],  # Way 2 - To Create a Tool for the Model/Agent
    system_prompt="You are a helpful assistant.",
)

agent

### Tool Execution Loop

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It’s sunny in Boston right now! Enjoy the clear skies.
